In [23]:
pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Triple quotes allow you to write the query on multiple lines
                                                                             # five %s use because you have five column

In [ ]:
# fetchhall to get all the rows # save the changes permanently database.

In [9]:

import pymysql

con = pymysql.connect(
    user="root",                                            # this all connect the sql to python
    host="localhost",
    database="attendance_db",
    password="12345"
)

cur = con.cursor()

while True:

    print("\n===== ATTENDANCE MANAGEMENT SYSTEM =====")
    print("1. Add Student")
    print("2. View Students")
    print("3. Mark Attendance")
    print("4. Batch-wise Attendance")
    print("5. Student Attendance Report")
    print("6. Update Student")
    print("7. Update Attendance")
    print("8. Add Marks")
    print("9. Exit")

    choice = input("Enter your choice: ")

    # 1. Add Student
    if choice == "1":

        while True:

            print("\nEnter Student Details")

            student_id = int(input("Student ID: "))
            student_name = input("Student Name: ")

            print("\n1. BCA")
            print("2. Civil")
            print("3. IT")
            print("4. AIML")

            batch_id = int(input("Select Batch ID: "))

            if batch_id == 1:
                branch = "BCA"
            elif batch_id == 2:
                branch = "Civil"
            elif batch_id == 3:
                branch = "IT"
            elif batch_id == 4:
                branch = "AIML"
            else:
                print("Invalid Batch ID!")
                continue

            print("Branch_name:", branch)

            phone = input("Phone Number: ")
            
    # Triple quotes allow you to write the query on multiple lines
            query = """
            INSERT INTO students
            (student_id, student_name, batch_id, branch, phone)
            VALUES (%s, %s, %s, %s, %s)
            """ 
            values = (student_id, student_name, batch_id, branch, phone)

            cur.execute(query, values)

            con.commit()                # save the changes permanently database.

            print("Student added successfully!")

            again = input("Add another student? (yes/no): ")

            if again.lower() == "no":
                print("\nStudent Entry Completed!")

        break

    # 2. View Students (Batch-wise)
    elif choice == "2":

        print("\n1. BCA")
        print("2. Civil")
        print("3. IT")
        print("4. AIML")

        batch_id = int(input("Select Batch ID: "))

        query = """
        SELECT student_id, student_name,
               batch_id, branch, phone, marks
        FROM students
        WHERE batch_id = %s
        ORDER BY student_id
        """
     # %s means placeholder for value
        
        cur.execute(query, (batch_id,))
        rows = cur.fetchall()                      # fetchhall to get all the rows

        if len(rows) == 0:
            print("No students found in this batch!")

        else:

            print("\nID | Name | Batch ID | Branch | Phone | Marks")

            for row in rows:
                print(row)

            print("\nAll Students Details Displayed!")

        break

    # 3. Mark Attendance
    elif choice == "3":

        print("\n1. BCA")
        print("2. Civil")
        print("3. IT")
        print("4. AIML")

        batch_id = int(input("Select Batch ID: "))
        attendance_date = input("Enter date (YYYY-MM-DD): ")

        query = """
        SELECT student_id, student_name
        FROM students
        WHERE batch_id = %s
        ORDER BY student_id
        """

        cur.execute(query, (batch_id,))
        students = cur.fetchall()

        if len(students) == 0:
            print("No students found in this batch!")

        else:

            for student in students:

                print("\nStudent ID:", student[0])        # query mai stud_id or stud_name tha toh id[0] and name[1]
                print("Student Name:", student[1])

                print("1. Present")
                print("2. Absent")
                print("3. Leave")

                status_choice = input("Select Status: ")

                if status_choice == "1":
                    status = "Present"
                elif status_choice == "2":
                    status = "Absent"
                elif status_choice == "3":
                    status = "leave"
                else:
                    print("Invalid status! Skipping student.")
                    continue

                query = """
                INSERT INTO attendance
                (student_id, attendance_date, status)
                VALUES (%s, %s, %s)
                """

                cur.execute(query, (
                    student[0], attendance_date, status
                ))

                con.commit()

                print("Attendance marked successfully!")

            print("\nAll Students Attendance Completed!")

        break

    # 4. Batch-wise Attendance
    elif choice == "4":

        print("\n1. BCA")
        print("2. Civil")
        print("3. IT")
        print("4. AIML")

        batch_id = int(input("Enter Batch ID: "))
        attendance_date = input("Enter Date (YYYY-MM-DD): ")

        query = """
        SELECT s.student_id, s.student_name,
               b.batch_name, a.attendance_date, a.status
        FROM students s
        JOIN batches b ON s.batch_id = b.batch_id
        LEFT JOIN attendance a
          ON s.student_id = a.student_id
          AND a.attendance_date = %s
        WHERE s.batch_id = %s
        ORDER BY s.student_id
        """

        cur.execute(query, (attendance_date, batch_id))
        rows = cur.fetchall()

        for row in rows:
            print(row)

        break

    # 5. Student Attendance Report
    elif choice == "5":

        student_id = int(input("Enter Student ID: "))

        query = """
        SELECT s.student_id, s.student_name,
               COUNT(a.attendance_id) AS total_days,
               SUM(a.status = 'Present') AS present_days,
               SUM(a.status = 'Absent') AS absent_days,
               SUM(a.status = 'leave') AS leave_days
        FROM students s
        LEFT JOIN attendance a ON s.student_id = a.student_id
        WHERE s.student_id = %s
        GROUP BY s.student_id, s.student_name
        """

        cur.execute(query, (student_id,))
        rows = cur.fetchall()

        for row in rows:

            print("\nStudent ID:", row[0])
            print("Student Name:", row[1])
            print("Total Attendance Records:", row[2])
            print("Present:", row[3] or 0)
            print("Absent:", row[4] or 0)
            print("Leave:", row[5] or 0)
            print("\nStudents Attendance Report!")

        break

    # 6. Update Student
    elif choice == "6":

        student_id = int(input("Enter Student ID: "))

        query = """
        SELECT student_id, student_name, batch_id, branch, phone
        FROM students
        WHERE student_id = %s
        """

        cur.execute(query, (student_id,))
        row = cur.fetchone()

        if row is None:
            print("Student Not Found!")

        else:
            print("\nCurrent Details:", row)

            student_name = input("Enter New Student Name: ")

            print("\n1. BCA")
            print("2. Civil")
            print("3. IT")
            print("4. AIML")

            batch_id = int(input("Enter New Batch ID: "))

            if batch_id == 1:
                branch = "BCA"
            elif batch_id == 2:
                branch = "Civil"
            elif batch_id == 3:
                branch = "IT"
            elif batch_id == 4:
                branch = "AIML"
            else:
                print("Invalid Batch ID!")
                break

            phone = input("Enter New Phone Number: ")

            query = """
            UPDATE students
            SET student_name = %s,
                batch_id = %s,
                branch = %s,
                phone = %s
            WHERE student_id = %s
            """

            cur.execute(query, (
                student_name, batch_id, branch,
                phone, student_id
            ))

            con.commit()

            print("\nStudent Updated Successfully!")

        break

    # 7. Update Attendance
    elif choice == "7":

        student_id = int(input("Enter Student ID: "))
        attendance_date = input("Enter Attendance Date (YYYY-MM-DD): ")

        query = """
        SELECT status
        FROM attendance
        WHERE student_id = %s
        AND attendance_date = %s
        """

        cur.execute(query, (student_id, attendance_date))
        row = cur.fetchone()

        if row is None:
            print("Attendance Record Not Found!")

        else:
            print("\nCurrent Attendance:", row[0])

            print("\n1. Present")
            print("2. Absent")
            print("3. Leave")

            status_choice = input("Enter New Status: ")

            if status_choice == "1":
                status = "Present"
            elif status_choice == "2":
                status = "Absent"
            elif status_choice == "3":
                status = "leave"
            else:
                print("Invalid Status!")
                break

            query = """
            UPDATE attendance
            SET status = %s
            WHERE student_id = %s
            AND attendance_date = %s
            """

            cur.execute(query, (
                status, student_id, attendance_date
            ))

            con.commit()

            print("\nAttendance Updated Successfully!")

        break

    # 8. Add Marks (Batch-wise)
    elif choice == "8":

        print("\n1. BCA")
        print("2. Civil")
        print("3. IT")
        print("4. AIML")

        batch_id = int(input("Select Batch ID: "))

        query = """
        SELECT student_id, student_name
        FROM students
        WHERE batch_id = %s
        ORDER BY student_id
        """

        cur.execute(query, (batch_id,))
        students = cur.fetchall()

        if len(students) == 0:
            print("No students found in this batch!")

        else:

            for student in students:

                print("\nStudent ID:", student[0])
                print("Student Name:", student[1])

                marks = int(input("Enter Student Marks: "))

                query = """
                UPDATE students
                SET marks = %s
                WHERE student_id = %s
                """

                cur.execute(query, (marks, student[0]))

                con.commit()

                print("Marks Added Successfully!")

            print("\nAll Students Marks Added Successfully!")

        break

    # 9. Exit
    elif choice == "9":

        print("Attendance Management System Closed.")
        break

    else:
        print("Invalid choice! Please try again.")

    break

cur.close()
con.close()


===== ATTENDANCE MANAGEMENT SYSTEM =====
1. Add Student
2. View Students
3. Mark Attendance
4. Batch-wise Attendance
5. Student Attendance Report
6. Update Student
7. Update Attendance
8. Add Marks
9. Exit


Enter your choice:  2



1. BCA
2. Civil
3. IT
4. AIML


Select Batch ID:  1



ID | Name | Batch ID | Branch | Phone | Marks
(3, 'rahul', 1, 'bca', '7678431211', 67)
(7, 'meena', 1, 'BCA', '7656456544', 54)
(8, 'sweety', 1, 'BCA', '8887654321', 78)
(12, 'reena', 1, 'BCA', '7956645678', 76)
(101, 'Anjali', 1, 'BCA', '9876543210', 67)
(105, 'aman', 1, 'BCA', '8890675467', 87)
(109, 'Arjun', 1, 'BCA', '9876543218', 76)
(113, 'Vikas', 1, 'BCA', '9876543222', 78)
(114, 'riya', 1, 'BCA', '9890767876', 76)

All Students Details Displayed!


In [1]:
!pip install pymysql